# 05 · Score every system on the locked test set
All calls cached in results/cache, so reruns are free. Requires FIREWORKS_DEPLOYMENT_MODEL.

In [ ]:
import sys, json, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from dotenv import load_dotenv; load_dotenv(ROOT / '.env')
import nest_asyncio; nest_asyncio.apply()
import asyncio

In [ ]:
import pt_systems as S, pt_score as SC
from pt_llm import LLM
llm = LLM(concurrency=16, cache_name='score')
rows = S.test_rows()
out = {}
for name, fn in [('base', S.base), ('harness_rubric', S.harness), ('sft', S.sft), ('sft_safety', S.sft_safety),
                 ('sft_router', S.sft_router), ('frontier', S.frontier)]:
    try:
        out[name] = await fn(llm, rows)
        print(name, json.dumps(SC.by_modality(out[name])['all']))
    except Exception as e:
        print(name, 'FAILED:', e)
# acceptance: identical prompt per run
print({n: {r['prompt_hash'] for r in v} for n, v in out.items()})